<a href="https://colab.research.google.com/github/thedroppedcroisssantttt/CAP5610-ML-Project/blob/main/ML_term_project_SZ_contributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#data import & prep

In [ ]:
!pip install datasets transformers scikit-learn torch accelerate -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.utils import shuffle

In [ ]:
from huggingface_hub import login
login()

In [ ]:
dataset = load_dataset("fancyzhx/dbpedia_14")

print(dataset)
print("\nexample entry:")
print(dataset['train'][0])

In [ ]:
label_names = dataset['train'].features['label'].names
print("number of classes:", len(label_names))
print("class labels:", label_names)

print(f"\ntraining split: {len(dataset['train'])}")
print(f"\ntesting split: {len(dataset['test'])}")

In [ ]:
#kernal svm

In [ ]:
from sklearn.utils import shuffle

def sample_dataset(hf_dataset, samples_per_class=500, seed=42):
    """Sample a fixed number of examples per class for balance."""
    df = pd.DataFrame(hf_dataset)
    sampled = df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(samples_per_class, len(x)), random_state=seed)
    )
    return shuffle(sampled, random_state=seed).reset_index(drop=True)

train_df = sample_dataset(dataset['train'], samples_per_class=5000)
test_df  = sample_dataset(dataset['test'],  samples_per_class=200)

print(f"sampled train size: {len(train_df)}")
print(f"sampled test size: {len(test_df)}")
print("\nclass distribution in train set:")
print(train_df['label'].value_counts().sort_index())

In [ ]:
train_df['text'] = train_df['title'] + " " + train_df['content']
test_df['text']  = test_df['title']  + " " + test_df['content']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidfVectorizer = TfidfVectorizer(max_features=50000, sublinear_tf=True)

X_train = tfidfVectorizer.fit_transform(train_df['text'])
X_test  = tfidfVectorizer.transform(test_df['text'])

y_train = train_df['label'].values
y_test  = test_df['label'].values

print("train shape:", X_train.shape)
print("test shape: ", X_test.shape)

In [ ]:
from sklearn.svm import SVC

svmModel = SVC(kernel='rbf', C=1.0, gamma='scale')
svmModel.fit(X_train, y_train)

print("training complete")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

y_pred = svmModel.predict(X_test)

svmAccuracy = accuracy_score(y_test, y_pred)
svmMacroF1  = f1_score(y_test, y_pred, average='macro')

print("accuracy: ", round(svmAccuracy, 4))
print("f1: ", round(svmMacroF1, 4))

In [ ]:
cmSVM = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cmSVM, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names,
            yticklabels=label_names)
plt.title("kernal svm confusion matrix")
plt.xlabel("pred.")
plt.ylabel("act.")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

if(torch.cuda.memory_allocated() == 0):
  print("mem clear")
else:
  print("mem not clear")

In [ ]:
#decoder only llm

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenizeData(df, tokenizer, maxLength=128):
    return tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=maxLength,
        return_tensors='pt'
    )

trainEncodings = tokenizeData(train_df, tokenizer)
testEncodings  = tokenizeData(test_df, tokenizer)

In [ ]:
print("input ids shape:", trainEncodings['input_ids'].shape)

In [ ]:
import torch
from torch.utils.data import Dataset

class DBpediaDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

trainDataset = DBpediaDataset(trainEncodings, y_train)
testDataset  = DBpediaDataset(testEncodings, y_test)

In [ ]:
from transformers import GPT2ForSequenceClassification

gptModel = GPT2ForSequenceClassification.from_pretrained(
    "distilgpt2",
    num_labels=14
)
gptModel.config.pad_token_id = tokenizer.eos_token_id

In [ ]:
from transformers import TrainingArguments

trainingArgs = TrainingArguments(
    output_dir='./gpt_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    fp16=True,
    report_to='none'
)

In [ ]:
from transformers import Trainer
from sklearn.metrics import accuracy_score, f1_score

def computeMetrics(evalPred):
    logits, labels = evalPred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro')
    }

trainer = Trainer(
    model=gptModel,
    args=trainingArgs,
    train_dataset=trainDataset,
    eval_dataset=testDataset,
    compute_metrics=computeMetrics
)

In [ ]:
trainer.train()

In [ ]:
gptPreds  = trainer.predict(testDataset)
gptLabels = gptPreds.label_ids
gptPredLabels = np.argmax(gptPreds.predictions, axis=-1)

gptAccuracy = accuracy_score(gptLabels, gptPredLabels)
gptMacroF1  = f1_score(gptLabels, gptPredLabels, average='macro')

print("accuracy:", round(gptAccuracy, 4))
print("f1:", round(gptMacroF1, 4))

In [ ]:
labelNames = dataset['train'].features['label'].names

cmGPT = confusion_matrix(gptLabels, gptPredLabels)

plt.figure(figsize=(12, 10))
sns.heatmap(cmGPT, annot=True, fmt='d', cmap='Purples',
            xticklabels=labelNames,
            yticklabels=labelNames)
plt.title("distilgpt2 confusion matrix")
plt.xlabel("pred.")
plt.ylabel("act.")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()